In [1]:
import os, sys
# if you start Jupyter from project root, this is not needed,
# but let's keep it here to be robust:
PROJECT_ROOT = "/Users/sko/src/LeelaX"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

import chess
import torch
import numpy as np

from leelax.env.encode import state_to_tensor
from leelax.env.policy_index import legal_policy_mask, move_to_index, index_to_move
from leelax.net.model import LeelaXNet
from leelax.net.inference import masked_softmax, select_action

In [2]:
# create a position
board = chess.Board()  # startpos
print(board)
print(board.fen())

state = state_to_tensor(board).unsqueeze(0)  # [1, 24, 8, 8]
mask_np = legal_policy_mask(board)           # (4672,)
mask = torch.from_numpy(mask_np).unsqueeze(0).float()  # [1, 4672]

state.shape, mask.shape

r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1


(torch.Size([1, 24, 8, 8]), torch.Size([1, 4672]))

In [3]:
net = LeelaXNet()
net.eval()

with torch.no_grad():
    policy_logits, value = net(state)

print("policy logits shape:", policy_logits.shape)
print("value:", value.item())

policy logits shape: torch.Size([1, 4672])
value: -0.016770150512456894


In [4]:
with torch.no_grad():
    probs = masked_softmax(policy_logits, mask, dim=1)  # [1, 4672]
    # greedy pick
    action_idx = select_action(policy_logits, mask, greedy=True).item()

print("chosen action index:", action_idx)

# convert back to move so we can read it:
mv = index_to_move(action_idx)
print("move:", mv.uci(), "| SAN:", board.san(mv))

chosen action index: 501
move: g1f3 | SAN: Nf3
